# Chapter 1 Companion — Mathematical Foundations

Runnable companions to the worked examples in
[`docs/01_mathematical_foundations`](../docs/01_mathematical_foundations):

- [`01_linear_algebra.md`](../docs/01_mathematical_foundations/01_linear_algebra.md)
- [`02_complex_numbers_and_hilbert_spaces.md`](../docs/01_mathematical_foundations/02_complex_numbers_and_hilbert_spaces.md)
- [`03_tensor_products_and_multipartite_systems.md`](../docs/01_mathematical_foundations/03_tensor_products_and_multipartite_systems.md)

Every number printed here is cross-checked against the docs' worked examples/exercises.

In [1]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)

# Shared constants: computational basis and Pauli matrices
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)

## Example 1 — Spectral decomposition of the Hadamard matrix

*Docs: `01_linear_algebra.md`, "Worked Example".* The Hadamard matrix is Hermitian **and**
unitary, so its eigenvalues must be real and lie on the unit circle: `λ = ±1`. The docs derive
eigenvectors `|v₊⟩ = cos(π/8)|0⟩ + sin(π/8)|1⟩ ≈ 0.924|0⟩ + 0.383|1⟩` and
`|v₋⟩ = sin(π/8)|0⟩ − cos(π/8)|1⟩`, and verify `H = (+1)|v₊⟩⟨v₊| + (−1)|v₋⟩⟨v₋|`.
We reproduce all of this numerically.

In [2]:
evals, evecs = np.linalg.eigh(H)
print("eigenvalues:", evals)                       # docs: λ = ±1

# docs give |v+> = (cos π/8, sin π/8), |v-> = (sin π/8, -cos π/8)
v_plus_doc  = np.array([np.cos(np.pi/8),  np.sin(np.pi/8)])
v_minus_doc = np.array([np.sin(np.pi/8), -np.cos(np.pi/8)])
print("cos(pi/8), sin(pi/8) =", np.cos(np.pi/8), np.sin(np.pi/8))   # docs: 0.924, 0.383

# eigh may return eigenvectors up to sign; compare overlap magnitudes
v_minus, v_plus = evecs[:, 0], evecs[:, 1]         # sorted ascending: -1 first
print("|<v+_doc | v+_numpy>| =", abs(v_plus_doc @ v_plus))
print("|<v-_doc | v-_numpy>| =", abs(v_minus_doc @ v_minus))

# Spectral decomposition: H = (+1)|v+><v+| + (-1)|v-><v-|
H_rebuilt = np.outer(v_plus_doc, v_plus_doc) - np.outer(v_minus_doc, v_minus_doc)
print("spectral decomposition reproduces H:", np.allclose(H_rebuilt, H))
print("H^2 = I (involution):", np.allclose(H @ H, np.eye(2)))
assert np.allclose(sorted(evals), [-1, 1])
assert np.allclose(H_rebuilt, H)

eigenvalues: [-1.  1.]
cos(pi/8), sin(pi/8) = 0.9238795325112867 0.3826834323650898
|<v+_doc | v+_numpy>| = 1.0
|<v-_doc | v-_numpy>| = 0.9999999999999999
spectral decomposition reproduces H: True
H^2 = I (involution): True


## Example 2 — Expanding a Hermitian matrix in the Pauli basis

*Docs: `01_linear_algebra.md`, Exercise 2.* `{I, X, Y, Z}` is an orthogonal basis for 2×2
Hermitian matrices under `⟨A,B⟩ = ½Tr(A†B)`, so each coefficient is a half-trace.
For `A = [[1,2],[2,−1]]` the docs find `a=0, b=2, c=0, d=1`, i.e. `A = 2X + Z`.

In [3]:
A = np.array([[1, 2], [2, -1]], dtype=complex)

coeffs = {name: 0.5 * np.trace(P.conj().T @ A).real
          for name, P in [("I", I2), ("X", X), ("Y", Y), ("Z", Z)]}
print("Pauli coefficients:", coeffs)               # docs: {I:0, X:2, Y:0, Z:1}

A_rebuilt = coeffs["I"]*I2 + coeffs["X"]*X + coeffs["Y"]*Y + coeffs["Z"]*Z
print("A = 2X + Z reproduces A:", np.allclose(A_rebuilt, A))
assert np.isclose(coeffs["X"], 2) and np.isclose(coeffs["Z"], 1)
assert np.isclose(coeffs["I"], 0) and np.isclose(coeffs["Y"], 0)

Pauli coefficients: {'I': np.float64(0.0), 'X': np.float64(2.0), 'Y': np.float64(0.0), 'Z': np.float64(1.0)}
A = 2X + Z reproduces A: True


## Example 3 — The √X gate: unitarity, `V² = X`, eigenvalues `{1, i}`

*Docs: `01_linear_algebra.md`, Exercise 4.* `V = ½[[1+i, 1−i],[1−i, 1+i]]` is the `SX` gate.
The docs show `V†V = I`, `V² = X`, and eigenvalues `{1, i}` — square roots of the eigenvalues
`{1, −1}` of `X`, on the eigenvectors `|±⟩` of `X`.

In [4]:
V = 0.5 * np.array([[1+1j, 1-1j], [1-1j, 1+1j]])

print("V is unitary:", np.allclose(V.conj().T @ V, np.eye(2)))
print("V^2 = X:", np.allclose(V @ V, X))

evals_V = np.linalg.eigvals(V)
print("eigenvalues of V:", np.round(evals_V, 6))    # docs: {1, i}

plus  = (ket0 + ket1) / np.sqrt(2)
minus = (ket0 - ket1) / np.sqrt(2)
print("V|+> = |+>  :", np.allclose(V @ plus, plus))
print("V|-> = i|-> :", np.allclose(V @ minus, 1j * minus))
assert np.allclose(sorted(evals_V, key=lambda z: z.imag), [1, 1j])

V is unitary: True
V^2 = X: True
eigenvalues of V: [ 1.+0.j -0.+1.j]
V|+> = |+>  : True
V|-> = i|-> : True


## Example 4 — Partial trace: Bell state and the W state

*Docs: `03_tensor_products_and_multipartite_systems.md`.* Two headline computations:

1. For the Bell state `|Φ⁺⟩ = (|00⟩+|11⟩)/√2` the reduced state of either qubit is the
   maximally mixed state `I/2` (section "Partial Trace").
2. For the three-qubit W state `|W⟩ = (|100⟩+|010⟩+|001⟩)/√3` the worked example finds
   `ρ₁ = diag(2/3, 1/3)` — qubit 1 is `|0⟩` with probability 2/3, `|1⟩` with probability 1/3.

In [5]:
def partial_trace_keep_first(rho, dim_keep, dim_rest):
    '''Trace out the trailing subsystem: rho on (keep ⊗ rest) -> reduced state on keep.'''
    rho = rho.reshape(dim_keep, dim_rest, dim_keep, dim_rest)
    return np.einsum("ajbj->ab", rho)

# 1) Bell state
bell = np.zeros(4, dtype=complex)
bell[0b00] = bell[0b11] = 1 / np.sqrt(2)
rho_bell = np.outer(bell, bell.conj())
rho_A = partial_trace_keep_first(rho_bell, 2, 2)
print("Bell reduced state of qubit A:\n", rho_A.real)   # docs: I/2
assert np.allclose(rho_A, np.eye(2) / 2)

# 2) W state (qubit 1 kept, qubits 2,3 traced out)
W = np.zeros(8, dtype=complex)
for idx in (0b100, 0b010, 0b001):
    W[idx] = 1 / np.sqrt(3)
rho_W = np.outer(W, W.conj())
rho_1 = partial_trace_keep_first(rho_W, 2, 4)
print("W-state reduced state of qubit 1:\n", rho_1.real)  # docs: diag(2/3, 1/3)
assert np.allclose(rho_1, np.diag([2/3, 1/3]))
print("mixed (Tr rho^2 < 1):", np.trace(rho_1 @ rho_1).real)

Bell reduced state of qubit A:
 [[0.5 0. ]
 [0.  0.5]]
W-state reduced state of qubit 1:
 [[0.666667 0.      ]
 [0.       0.333333]]
mixed (Tr rho^2 < 1): 0.5555555555555559


## Example 5 — Purity as an entanglement witness

*Docs: `03_tensor_products_and_multipartite_systems.md`, Exercise 4.* For
`|ψ⟩ = (|00⟩+|01⟩+|11⟩)/√3` the docs compute the reduced state of qubit B,
`ρ_B = (1/3)[[1,1],[1,2]]`, purity `Tr(ρ_B²) = 7/9 < 1` (hence entangled),
eigenvalues `(3±√5)/6 ≈ 0.873, 0.127`, and entanglement entropy `S ≈ 0.55` ebits.

In [6]:
psi = np.zeros(4, dtype=complex)
for idx in (0b00, 0b01, 0b11):
    psi[idx] = 1 / np.sqrt(3)

rho = np.outer(psi, psi.conj())
# reduced state of qubit B: trace out the *first* qubit
rho_B = rho.reshape(2, 2, 2, 2)
rho_B = np.einsum("iaib->ab", rho_B)
print("rho_B =\n", 3 * rho_B.real, "/ 3")           # docs: (1/3)[[1,1],[1,2]]
assert np.allclose(rho_B, np.array([[1, 1], [1, 2]]) / 3)

purity = np.trace(rho_B @ rho_B).real
print("purity Tr(rho_B^2) =", purity, "= 7/9 ->", np.isclose(purity, 7/9))

lam = np.linalg.eigvalsh(rho_B)
print("eigenvalues:", lam, " docs: (3∓√5)/6 =", (3-np.sqrt(5))/6, (3+np.sqrt(5))/6)
S = -sum(l * np.log2(l) for l in lam)
print("entanglement entropy S = %.4f ebits (docs: ≈ 0.55)" % S)
assert np.isclose(purity, 7/9)
assert 0.54 < S < 0.56

rho_B =
 [[1. 1.]
 [1. 2.]] / 3
purity Tr(rho_B^2) = 0.7777777777777782 = 7/9 -> True
eigenvalues: [0.127322 0.872678]  docs: (3∓√5)/6 = 0.12732200375003502 0.872677996249965
entanglement entropy S = 0.5500 ebits (docs: ≈ 0.55)
